In [4]:
import sys
from pathlib import Path

chapter_dir = str(Path("../programmingbitcoin/book/code-ch09").resolve())
if chapter_dir not in sys.path:
    sys.path.insert(0, chapter_dir)

from helper import hash256, little_endian_to_int


<span id="calculating-target" data-learnt="2026-11-19" data-topic="Proof-of-work, target & difficulty"></span>Block header


In [15]:
block_height = 482738
block_header_hex = (
    "020000208ec39428b17323fa0ddec8e887b4a7c53b8c0a0a220cfd000000000000000000"
    "5b0750fce0a889502d40508d39576821155e9c9e3f5c3157f961db38fd8b25be1e77a759"
    "e93c0118a4ffd71d"
)

header = bytes.fromhex(block_header_hex)
bits = header[72:76]
print("Bits: Compact target:", bits.hex())


Bits: Compact target: e93c0118


From here we have **coefficient: `e9 3c 01`.**
These stored bytes are in little-endian order.
So, the coefficient written as a hexadecimal number is `0x013ce9`.

And exponent: `18`

In [6]:
coefficient = little_endian_to_int(bits[:-1])
exponent = bits[-1]

print("Coefficient:", coefficient)
print("Exponent:", exponent)


Coefficient: 81129
Exponent: 24


Then we can do:

In [7]:
target = coefficient * 256**(exponent - 3)
print(f"{target:064x}")


0000000000000000013ce9000000000000000000000000000000000000000000


So, a miner has to create a hash for this block which should be lesser than `0000000000000000013ce9000000000000000000000000000000000000000000`. For this block, we can calculate the proof from the hash:

In [8]:
proof = little_endian_to_int(hash256(bytes.fromhex(block_header_hex)))

print(f"Target: {target:064x}")
print(f"Proof:  {proof:064x}")
print("Proof < Target:", proof < target)

Target: 0000000000000000013ce9000000000000000000000000000000000000000000
Proof:  0000000000000000007e9e4c586439b0cdbe13b1370bdd9435d76a644d047523
Proof < Target: True


We can also calculate what the difficulty would have been at this stage, via:

In [9]:
difficulty = 0xffff * 256**(0x1d-3) / target
print(f"Difficulty:  {difficulty}")

Difficulty:  888171856257.3206


Lets compares these to the latest [bitcoin block](https://mempool.space/block/00000000000000000001edd07eeb2d4b2319da51867db070bdba3e4b6cd1609a) today (Nov 20, 2026): 

In [21]:
latest_block_height = 976707
latest_block_header_hex = (
    "00000020b50942d0a6fea7ccd460af1260a376de2677b21ea54c00000000000000000000"
    "c60459c99c9dd0951e7551863329e92912fef8e95c8d41d49806715f1bdff03be500006b"
    "731602177e796b36"
)

latest_block_header = bytes.fromhex(latest_block_header_hex)
latest_block_bits = latest_block_header[72:76]
print("Bits: Compact target:", latest_block_bits.hex())

latest_block_coefficient = little_endian_to_int(latest_block_bits[:-1])
latest_block_exponent = latest_block_bits[-1]
print("Coefficient:", latest_block_coefficient)
print("Exponent:", latest_block_exponent)

latest_block_target = latest_block_coefficient * 256**(latest_block_exponent - 3)
latest_block_proof = little_endian_to_int(hash256(bytes.fromhex(latest_block_header_hex)))
print(f"Target: {latest_block_target:064x}")
print(f"Proof:  {latest_block_proof:064x}")
print("Proof < Target:", latest_block_proof < latest_block_target)

latest_block_difficulty = 0xffff * 256**(0x1d-3) / latest_block_target
print(f"Difficulty:  {latest_block_difficulty}")

Bits: Compact target: 73160217
Coefficient: 136819
Exponent: 23
Target: 0000000000000000000216730000000000000000000000000000000000000000
Proof:  00000000000000000001edd07eeb2d4b2319da51867db070bdba3e4b6cd1609a
Proof < Target: True
Difficulty:  134823837323272.66


So, we can see that the difficulty has risen by: